# Riemannian Geometry Diagnostic Battery

## Goal

Confirm or refute the paper's central Riemannian claim: that conservative
SPLM-family models induce a well-defined **Jacobi metric** on the hidden-state
manifold, and that inference trajectories approximately follow **geodesics** of
that metric.

## Models Under Test

| Model | Checkpoint | PPL | Conservativity |
|-------|-----------|-----|----------------|
| Multi-Xi SPLM | `dimitarpg13/semsimula-splm-multixi` | 11.51 | Fully conservative |
| Fock v2.1 PARFLM | `dimitarpg13/semsimula-fock-parflm` | 9.30 | Conservative + controlled Q_i |
| Fock Attention PARFLM | `dimitarpg13/semsimula-fock-attention` | 9.42 | Conservative + direct exchange |

## Arms

| # | Arm | Tests |
|---|-----|-------|
| 1 | Metric validity | Conformal factor Ω² > 0 (Jacobi metric positive-definite) |
| 2 | Geodesic compliance | Jacobi residual: observed vs predicted acceleration |
| 3 | Curvature proxy | K_max from Hessian of V_θ; correlation with logit entropy |
| 4 | Energy conservation | Mechanical energy H = T + V across layers; work decomposition |
| 5 | Conservativity separator | Shared-potential R² positioning on conservativity spectrum |

## Predicted Ordering (Confirm/Refute)

- Geodesic compliance: **SPLM >> Fock v2.1 > Fock Attention**
- Energy conservation: **SPLM (tight) >> Fock (controlled drift)**
- Conservativity R²: **SPLM (~0.95) > Fock v2.1 (~0.6-0.7) > Fock Attention**

In [ ]:
# ── Cell 1: Environment setup ──────────────────────────────────────
import subprocess, sys, os

def run(cmd):
    print(f"$ {cmd}")
    subprocess.check_call(cmd, shell=True)

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    run('pip install -q transformers huggingface_hub pyarrow scipy scikit-learn')
    if not os.path.isdir('semsimula-paper'):
        run('git clone --depth 1 https://github.com/dimitarpg13/semsimula-paper.git')
    REPO = 'semsimula-paper'
else:
    REPO = os.environ.get('SEMSIMULA_PAPER', '.')

ARCH_DIR = os.path.join(REPO, 'notebooks', 'conservative_arch')
for p in [
    ARCH_DIR,
    os.path.join(ARCH_DIR, 'multixi'),
    os.path.join(ARCH_DIR, 'parf'),
    os.path.join(ARCH_DIR, 'energetic_minima'),
    os.path.join(ARCH_DIR, 'sarf_mass_variant'),
    os.path.join(ARCH_DIR, 'scaleup'),
]:
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
import numpy as np

DEVICE = (
    'cuda' if torch.cuda.is_available()
    else ('mps' if hasattr(torch.backends, 'mps')
          and torch.backends.mps.is_available() else 'cpu')
)
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    print('TF32 disabled for autograd.grad stability')

In [ ]:
# ── Cell 2: Download checkpoints & define model registry ──────────
# Models are loaded ONE AT A TIME during each arm to keep peak RAM low.
# Only checkpoint paths and the loader function are set up here.

from huggingface_hub import hf_hub_download
import gc
from dataclasses import fields as dc_fields

CKPT_DIR = 'checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

HF_CKPTS = {
    'splm':      ('dimitarpg13/semsimula-splm-multixi',  'checkpoint/model_16k.pt'),
    'fock_v21':  ('dimitarpg13/semsimula-fock-parflm',   'checkpoint/model.pt'),
    'fock_attn': ('dimitarpg13/semsimula-fock-attention', 'checkpoint/model.pt'),
}

ckpt_paths = {}
for name, (repo, fname) in HF_CKPTS.items():
    print(f'Downloading {name} from {repo}...')
    path = hf_hub_download(repo_id=repo, filename=fname, local_dir=CKPT_DIR)
    ckpt_paths[name] = path
    print(f'  -> {path}')

# Precompute logfreq surprisal if not cached
SCRIPTS_DIR = os.path.join(ARCH_DIR, 'scaleup')
LOGFREQ_PATH = os.path.join(SCRIPTS_DIR, 'results', 'logfreq_surprisal_tinystories.npy')
if not os.path.exists(LOGFREQ_PATH):
    print('Computing logfreq surprisal (one-time, ~2 min)...')
    os.makedirs(os.path.dirname(LOGFREQ_PATH), exist_ok=True)
    subprocess.run(
        [sys.executable, os.path.join(SCRIPTS_DIR, 'compute_unigram_frequencies_tinystories.py')],
        cwd=SCRIPTS_DIR, check=True,
    )
    assert os.path.exists(LOGFREQ_PATH)
    print('Done.')
else:
    print(f'logfreq file exists: {LOGFREQ_PATH}')


def _load_from_ckpt(ckpt_path, ConfigClass, ModelClass, logfreq_path):
    """Load one model from checkpoint. Caller must del + free_mem() when done."""
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    cfg_dict = ckpt.get('config') or ckpt.get('model_cfg') or ckpt.get('cfg', {})
    if isinstance(cfg_dict, dict):
        known = {f.name for f in dc_fields(ConfigClass)}
        cfg = ConfigClass(**{k: v for k, v in cfg_dict.items() if k in known})
    else:
        cfg = cfg_dict
    if hasattr(cfg, 'logfreq_path'):
        cfg.logfreq_path = logfreq_path
    model = ModelClass(cfg)
    sd = ckpt.get('model_state_dict') or ckpt.get('state_dict') or ckpt
    model.load_state_dict(sd, strict=False)
    del ckpt, sd
    return model, cfg


def free_mem(model=None):
    """Unload a model and aggressively free GPU + CPU memory."""
    if model is not None:
        model.cpu()
        del model
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()


# Import model classes once so they're available in all arm cells
from model_multixi import (
    ScalarPotentialLMSARFMassLNMultiXi,
    SPLMSARFMassLNMultiXiConfig,
)
from model_fock_parf_multixi import FockMultiXiPARFLM, FockMultiXiPARFConfig
from model_fock_attention import FockAttentionPARFLM, FockAttentionConfig

# Model registry: name -> (ckpt_key, ConfigClass, ModelClass, is_splm)
MODEL_REGISTRY = {
    'Multi-Xi SPLM': ('splm',      SPLMSARFMassLNMultiXiConfig, ScalarPotentialLMSARFMassLNMultiXi, True),
    'Fock v2.1':     ('fock_v21',  FockMultiXiPARFConfig,       FockMultiXiPARFLM,                  False),
    'Fock Attention':('fock_attn', FockAttentionConfig,          FockAttentionPARFLM,                False),
}
MODEL_NAMES = list(MODEL_REGISTRY.keys())
print(f'\nRegistry ready: {MODEL_NAMES}')
print('Models will be loaded ONE AT A TIME per arm to minimise peak RAM.')

In [ ]:
# ── Cell 3: Load validation data & shared helpers ─────────────────
from data_module import load_tiny_stories, get_batch

# Use small max_train_tokens: we only need val data but train_ids is
# returned too and eats system RAM on T4 Colab (~12 GB sys RAM).
train_ids, val_ids = load_tiny_stories(max_train_tokens=500_000)
print(f'Validation tokens: {len(val_ids):,}')
del train_ids
gc.collect()

N_BATCHES = 3
BATCH_SIZE = 4
BLOCK_SIZE = 128
rng = np.random.default_rng(42)

batches_x_cpu = []
for _ in range(N_BATCHES):
    xb, _ = get_batch(val_ids, BATCH_SIZE, BLOCK_SIZE, rng)
    batches_x_cpu.append(torch.tensor(xb))   # CPU only

def get_batch_x(bi):
    return batches_x_cpu[bi].to(DEVICE)

print(f'Prepared {N_BATCHES} batches × {BATCH_SIZE} seqs × T={BLOCK_SIZE} (CPU)')
del val_ids
gc.collect()
print(f'System RAM after data prep: ~{os.popen("free -m 2>/dev/null || echo NA").read().strip()}')


def get_mass(model, x, is_splm):
    """Per-token mass, handling SPLM (x, emb) vs PARFLM (x) signatures.
    Always returns a CPU tensor (or Python float for scalar mass)."""
    if is_splm:
        emb = model._embed(x)
        m = model.compute_mass(x, emb)
    else:
        m = model.compute_mass(x)
    if isinstance(m, torch.Tensor):
        return m.detach().cpu()
    return m   # scalar


def extract_trajectory(model, x, is_splm):
    """L+1 hidden-state tensors moved to CPU, plus logits on CPU."""
    with torch.no_grad():
        if is_splm:
            out = model(x, targets=None, return_trajectory=True,
                        return_xi_trajectory=False)
        else:
            out = model(x, targets=None, return_trajectory=True)
        logits, _loss, traj = out[0], out[1], out[2]
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    traj_cpu = [h.detach().cpu() for h in traj]
    logits_cpu = logits.detach().cpu()
    del traj, out, logits
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return traj_cpu, logits_cpu


def load_model(name):
    """Load one model in float32 on DEVICE. Returns (model, is_splm)."""
    ckpt_key, ConfigClass, ModelClass, is_splm = MODEL_REGISTRY[name]
    model, _ = _load_from_ckpt(ckpt_paths[ckpt_key], ConfigClass, ModelClass, LOGFREQ_PATH)
    model.to(DEVICE).eval()
    n = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'  Loaded {name}: {n:.2f}M params (~{n * 4:.0f} MB VRAM)')
    return model, is_splm


print('Helpers ready. Running smoke test...')

# Smoke test: load first model, do a tiny forward pass, verify trajectory, unload
_test_name = MODEL_NAMES[0]
print(f'  Smoke: loading {_test_name}...')
_model, _is_splm = load_model(_test_name)
_x = get_batch_x(0)
print(f'  Smoke: forward pass...')
try:
    _traj, _logits = extract_trajectory(_model, _x, _is_splm)
    print(f'  Smoke: OK — {len(_traj)} layers, logits {tuple(_logits.shape)}')
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
        print(f'  Smoke: CUDA sync OK, '
              f'{torch.cuda.memory_allocated()/1e6:.0f} MB allocated, '
              f'{torch.cuda.max_memory_allocated()/1e6:.0f} MB peak')
    del _traj, _logits
except Exception as e:
    print(f'  Smoke FAILED: {type(e).__name__}: {e}')
del _x
free_mem(_model); del _model
print('Smoke test passed. Ready for arm cells.')

In [ ]:
# ── Cell 4: Arm 1 — Metric Validity ───────────────────────────────
# g_ij(h) = Ω²·δ_ij  where  Ω² = 2(E - V_θ)·m = m²·||v||²
# Positive whenever v ≠ 0 — sanity check + magnitude characterisation.
# Memory: light (forward-pass only). One model loaded at a time.

import matplotlib.pyplot as plt

print('═' * 60)
print('ARM 1: Metric Validity (Jacobi conformal factor Ω²)')
print('═' * 60)

arm1_results = {}

for name in MODEL_NAMES:
    print(f'\n── {name} ──')
    model, is_splm = load_model(name)
    all_frac, all_omega_mean, all_omega_std = [], [], []
    all_ke_mean, all_vt_mean = [], []

    x = get_batch_x(0)
    traj, _ = extract_trajectory(model, x, is_splm)   # traj is on CPU
    L = len(traj) - 1

    with torch.no_grad():
        m = get_mass(model, x, is_splm)   # already on CPU
        m_flat = m.squeeze(-1) if isinstance(m, torch.Tensor) and m.dim() > 2 else m
    del x
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

    for ell in range(1, L + 1):
        h_curr = traj[ell].to(DEVICE)
        h_prev = traj[ell - 1].to(DEVICE)
        v = h_curr - h_prev
        v_norm2 = (v ** 2).sum(dim=-1).cpu()
        del h_prev

        if isinstance(m_flat, torch.Tensor) and m_flat.dim() == 2:
            mf = m_flat
        else:
            mf = (m_flat.expand(v_norm2.shape) if isinstance(m_flat, torch.Tensor)
                  else torch.full_like(v_norm2, float(m_flat)))

        omega2 = (mf ** 2) * v_norm2
        ke = 0.5 * mf * v_norm2

        with torch.no_grad():
            xis = model.xi_module(h_curr.detach())
            vt = model.V_theta(xis, h_curr).squeeze(-1).cpu()
        del h_curr, xis

        all_frac.append((omega2 > 1e-12).float().mean().item())
        all_omega_mean.append(omega2.mean().item())
        all_omega_std.append(omega2.std().item())
        all_ke_mean.append(ke.mean().item())
        all_vt_mean.append(vt.mean().item())

    del traj
    arm1_results[name] = {
        'frac_positive': all_frac,
        'omega2_mean':   all_omega_mean,
        'omega2_std':    all_omega_std,
        'ke_mean':       all_ke_mean,
        'vt_mean':       all_vt_mean,
    }
    print(f'  Ω²>0 fraction: {["{:.4f}".format(f) for f in all_frac]}')
    print(f'  Mean Ω²:       {["{:.3e}".format(v) for v in all_omega_mean]}')
    print(f'  Mean KE:       {["{:.3e}".format(v) for v in all_ke_mean]}')
    print(f'  Mean V_θ:      {["{:.3e}".format(v) for v in all_vt_mean]}')
    free_mem(model); del model

# ── Plot ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for name, res in arm1_results.items():
    layers = range(1, len(res['frac_positive']) + 1)
    axes[0].plot(layers, res['frac_positive'], 'o-', label=name, markersize=4)
    axes[1].plot(layers, res['omega2_mean'],   'o-', label=name, markersize=4)
    axes[2].plot(layers, res['ke_mean'],        'o-', label=name, markersize=4)

axes[0].set_ylabel('Fraction Ω² > 0'); axes[0].set_xlabel('Layer')
axes[0].set_title('Metric Validity'); axes[0].legend(fontsize=8)
axes[0].set_ylim(0.9, 1.01)
axes[1].set_ylabel('Mean Ω²'); axes[1].set_xlabel('Layer')
axes[1].set_title('Conformal Factor Magnitude'); axes[1].legend(fontsize=8)
axes[2].set_ylabel('Mean KE'); axes[2].set_xlabel('Layer')
axes[2].set_title('Kinetic Energy Profile'); axes[2].legend(fontsize=8)
for ax in axes: ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()
print('\nArm 1 complete. ✓')

In [ ]:
# ── Cell 5: Arm 2 — Geodesic Compliance (Jacobi Residual) ─────────
# a_Jacobi^k = [2 v^k ⟨∇V,v⟩ − ||v||² (∂^k V)] / [2T]
# Compliance = 1 − ||a_obs − a_Jacobi||² / ||a_obs||²
# Memory: medium (autograd per layer). One model loaded at a time.

print('═' * 60)
print('ARM 2: Geodesic Compliance (Jacobi Residual)')
print('═' * 60)

arm2_results = {}


def compute_grad_V(model, h):
    """∇_h V_θ(ξ(h_detach), h) via autograd. Returns (B, T, d)."""
    h_in = h.detach().requires_grad_(True)
    xis = model.xi_module(h_in.detach())
    V = model.V_theta(xis, h_in)
    grad_V = torch.autograd.grad(V.sum(), h_in, create_graph=False)[0]
    return grad_V.detach()


for name in MODEL_NAMES:
    print(f'\n── {name} ──')
    model, is_splm = load_model(name)
    n_eval = min(N_BATCHES, 3)
    compliance_accum = None

    for bi in range(n_eval):
        x = get_batch_x(bi)
        traj, _ = extract_trajectory(model, x, is_splm)   # traj on CPU
        L = len(traj) - 1

        if compliance_accum is None:
            compliance_accum = [[] for _ in range(L - 1)]

        with torch.no_grad():
            m = get_mass(model, x, is_splm)   # already on CPU
            m_flat = m.squeeze(-1) if isinstance(m, torch.Tensor) and m.dim() > 2 else m
        del x

        for ell in range(1, L):
            h_prev = traj[ell - 1].to(DEVICE)
            h_curr = traj[ell].to(DEVICE)
            h_next = traj[ell + 1].to(DEVICE)

            v = h_curr - h_prev
            a_obs = h_next - 2 * h_curr + h_prev
            del h_prev, h_next

            grad_V = compute_grad_V(model, h_curr)

            v_norm2 = (v ** 2).sum(dim=-1, keepdim=True)
            mf = (m_flat.unsqueeze(-1).to(DEVICE)
                  if isinstance(m_flat, torch.Tensor) and m_flat.dim() == 2
                  else m_flat)
            ke = 0.5 * mf * v_norm2
            denom = (2.0 * ke).clamp(min=1e-8)

            gV_dot_v = (grad_V * v).sum(dim=-1, keepdim=True)
            a_jacobi = (2.0 * v * gV_dot_v - v_norm2 * grad_V) / denom

            R2 = ((a_obs - a_jacobi) ** 2).sum(dim=-1).mean().item()
            a2 = (a_obs ** 2).sum(dim=-1).mean().item()
            compliance_accum[ell - 1].append(1.0 - R2 / max(a2, 1e-12))
            del h_curr, v, a_obs, grad_V, a_jacobi

        del traj
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

    mean_comp = [np.mean(c) for c in compliance_accum]
    arm2_results[name] = {
        'compliance': mean_comp,
        'mean_compliance': float(np.mean(mean_comp)),
    }
    print(f'  Per-layer compliance: {["{:.4f}".format(c) for c in mean_comp]}')
    print(f'  Mean compliance: {np.mean(mean_comp):.4f}')
    free_mem(model); del model

# ── Plot ──
fig, ax = plt.subplots(figsize=(8, 4))
for name, res in arm2_results.items():
    L = len(res['compliance'])
    ax.plot(range(2, L + 2), res['compliance'], 'o-', label=name, markersize=5)
ax.set_ylabel('Geodesic Compliance (R²)'); ax.set_xlabel('Layer')
ax.set_title('Jacobi Residual — Compliance', fontweight='bold')
ax.axhline(0, color='gray', linestyle='--', alpha=0.4)
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()
print('\nArm 2 complete. ✓')

In [ ]:
# ── Cell 6: Arm 3 — Curvature Proxy ───────────────────────────────
# K_max = λ_max(∇²V_θ) / [2T]  via HVP power iteration.
# Memory: heavy (create_graph=True per position). One model at a time;
# autograd graph explicitly freed after each position.

from scipy.stats import spearmanr

print('═' * 60)
print('ARM 3: Curvature Proxy (K_max from Hessian of V_θ)')
print('═' * 60)

N_POWER_ITER = 20
N_SAMPLE_POS = 32
N_SAMPLE_SEQ = 3

arm3_results = {}


def lambda_max_hvp(model, h_single, xi_single, n_iter=20):
    """λ_max of Hess_h(V_θ) at one position via power iteration.

    h_single: (d,)   xi_single: (K, d)
    Graph is freed after every iteration — no graph accumulation.
    """
    d = h_single.shape[0]
    u = torch.randn(d, device=h_single.device, dtype=h_single.dtype)
    u = u / u.norm()
    lam = 0.0

    for _ in range(n_iter):
        h_in = h_single.detach().requires_grad_(True)
        xi_in = xi_single.unsqueeze(0).unsqueeze(0)   # (1,1,K,d)
        h_3d = h_in.unsqueeze(0).unsqueeze(0)          # (1,1,d)
        V = model.V_theta(xi_in, h_3d)
        gV = torch.autograd.grad(V.sum(), h_in, create_graph=True)[0]
        Hv = torch.autograd.grad(
            (gV * u.detach()).sum(), h_in, retain_graph=False
        )[0]
        lam = (Hv.detach() @ u).item()
        u = Hv.detach()
        u = u / u.norm().clamp(min=1e-10)
        del h_in, V, gV, Hv   # explicit free each iter

    return abs(lam)


for name in MODEL_NAMES:
    print(f'\n── {name} ──')
    model, is_splm = load_model(name)
    kmax_list, ent_list = [], []

    x = get_batch_x(0)
    traj, logits = extract_trajectory(model, x, is_splm)   # traj on CPU
    L = len(traj) - 1
    mid = L // 2

    probs = torch.softmax(logits.float(), dim=-1)
    entropy = -(probs * torch.log(probs + 1e-10)).sum(dim=-1)   # (B, T) CPU
    del logits, probs

    # Compute xis and KE on GPU, then move everything to CPU
    h_mid_gpu  = traj[mid].to(DEVICE)
    h_prev_gpu = traj[max(mid - 1, 0)].to(DEVICE)
    del traj

    with torch.no_grad():
        m = get_mass(model, x, is_splm)   # already on CPU
        m_flat = m.squeeze(-1) if isinstance(m, torch.Tensor) and m.dim() > 2 else m
        xis_mid = model.xi_module(h_mid_gpu.detach()).detach().cpu()
        v = h_mid_gpu - h_prev_gpu
        v_norm2 = (v ** 2).sum(dim=-1).cpu()
        del h_prev_gpu, v
        if isinstance(m_flat, torch.Tensor) and m_flat.dim() == 2:
            mf = m_flat
        else:
            mf = (m_flat.expand(v_norm2.shape) if isinstance(m_flat, torch.Tensor)
                  else torch.full_like(v_norm2, float(m_flat)))
        E_minus_V = (0.5 * mf * v_norm2).cpu()   # KE on CPU

    h_mid_cpu = h_mid_gpu.detach().cpu()
    del h_mid_gpu, x
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

    B, T, d = h_mid_cpu.shape
    t_idx = np.random.choice(T, min(N_SAMPLE_POS, T), replace=False)

    for b in range(min(B, N_SAMPLE_SEQ)):
        for t in t_idx:
            h_pos = h_mid_cpu[b, t].to(DEVICE)
            xi_pos = xis_mid[b, t].to(DEVICE)
            lam = lambda_max_hvp(model, h_pos, xi_pos, n_iter=N_POWER_ITER)
            del h_pos, xi_pos
            ev = E_minus_V[b, t].item()
            kmax_list.append(lam / max(2 * abs(ev), 1e-10))
            ent_list.append(entropy[b, t].item())
        print(f'  seq {b+1}/{min(B, N_SAMPLE_SEQ)} done')
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

    del h_mid_cpu, xis_mid
    kmax_arr = np.array(kmax_list)
    ent_arr  = np.array(ent_list)
    rho, pval = spearmanr(kmax_arr, ent_arr) if len(kmax_arr) > 2 else (0., 1.)

    arm3_results[name] = {
        'kmax': kmax_arr.tolist(), 'entropy': ent_arr.tolist(),
        'kmax_mean': float(kmax_arr.mean()), 'kmax_std': float(kmax_arr.std()),
        'spearman_rho': float(rho), 'spearman_pval': float(pval),
    }
    print(f'  K_max: mean={kmax_arr.mean():.4e}, std={kmax_arr.std():.4e}')
    print(f'  Spearman ρ(K_max, H_softmax) = {rho:.4f}  (p={pval:.4e})')
    free_mem(model); del model

# ── Plot ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
palette = {'Multi-Xi SPLM': '#3B82F6', 'Fock v2.1': '#22C55E', 'Fock Attention': '#F59E0B'}
for i, (name, res) in enumerate(arm3_results.items()):
    axes[i].scatter(res['kmax'], res['entropy'], alpha=0.5, s=12,
                    color=palette[name])
    axes[i].set_xlabel('K_max'); axes[i].set_ylabel('Logit entropy')
    axes[i].set_title(f'{name}\nρ={res["spearman_rho"]:.3f}', fontsize=10)
    axes[i].spines['top'].set_visible(False); axes[i].spines['right'].set_visible(False)
plt.suptitle('Curvature Proxy vs. Prediction Uncertainty', fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()
print('\nArm 3 complete. ✓')

In [ ]:
# ── Cell 7: Arm 4 — Energy Conservation Profile ───────────────────
# Track H_l = T + V across layers; decompose ΔH for Fock models.
# W_damp ≈ −γ·||v||²·dt;  W_exchange = ΔH − W_damp.
# Memory: light (forward-pass only). One model at a time.

print('═' * 60)
print('ARM 4: Energy Conservation Profile')
print('═' * 60)

arm4_results = {}

for name in MODEL_NAMES:
    print(f'\n── {name} ──')
    model, is_splm = load_model(name)
    n_eval = min(N_BATCHES, 3)

    energy_accum = None
    w_damp_accum = None

    for bi in range(n_eval):
        x = get_batch_x(bi)
        traj, _ = extract_trajectory(model, x, is_splm)   # traj on CPU
        L = len(traj) - 1

        if energy_accum is None:
            energy_accum = [[] for _ in range(L + 1)]
            w_damp_accum = [[] for _ in range(L)]

        with torch.no_grad():
            m = get_mass(model, x, is_splm)   # already on CPU
            m_flat = m.squeeze(-1) if isinstance(m, torch.Tensor) and m.dim() > 2 else m
            gamma = model.gamma.item() if hasattr(model, 'gamma') and hasattr(model.gamma, 'item') else (float(model.gamma) if hasattr(model, 'gamma') else 0.0)
            dt = getattr(getattr(model, 'cfg', None), 'dt', 1.0)
        del x

        for ell in range(L + 1):
            h_curr = traj[ell].to(DEVICE)
            h_prev = traj[ell - 1].to(DEVICE) if ell > 0 else None
            v = (h_curr - h_prev) if h_prev is not None else torch.zeros_like(h_curr)
            del h_prev

            v_norm2 = (v ** 2).sum(dim=-1).cpu()
            if isinstance(m_flat, torch.Tensor) and m_flat.dim() == 2:
                mf = m_flat
            else:
                mf = (m_flat.expand(v_norm2.shape) if isinstance(m_flat, torch.Tensor)
                      else torch.full_like(v_norm2, float(m_flat)))

            with torch.no_grad():
                xis = model.xi_module(h_curr.detach())
                V_val = model.V_theta(xis, h_curr).squeeze(-1).cpu()
            del h_curr, xis, v

            ke = 0.5 * mf * v_norm2
            energy_accum[ell].append((ke + V_val).mean().item())

            if ell > 0:
                w_damp_accum[ell - 1].append(-gamma * v_norm2.mean().item() * dt)

        del traj
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

    mean_energy = [np.mean(e) for e in energy_accum]
    mean_w_damp = [np.mean(w) for w in w_damp_accum]
    delta_h     = [mean_energy[i+1] - mean_energy[i] for i in range(len(mean_energy)-1)]
    abs_delta_h = [abs(d) for d in delta_h]
    w_exchange  = [delta_h[i] - mean_w_damp[i] for i in range(len(delta_h))]
    total_drift = abs(mean_energy[-1] - mean_energy[0])
    rel_drift   = total_drift / max(abs(mean_energy[0]), 1e-10)

    arm4_results[name] = {
        'energy_profile': mean_energy, 'delta_h': abs_delta_h,
        'w_damp': mean_w_damp, 'w_exchange': w_exchange,
        'total_drift': total_drift, 'relative_drift': rel_drift,
    }
    print(f'  H(l): {["{:.4f}".format(e) for e in mean_energy]}')
    print(f'  Total drift: {total_drift:.6f}  ({rel_drift*100:.2f}%)')
    print(f'  W_damp:     {["{:.6f}".format(w) for w in mean_w_damp]}')
    print(f'  W_exchange: {["{:.6f}".format(w) for w in w_exchange]}')
    free_mem(model); del model

# ── Plot ──
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for name, res in arm4_results.items():
    L = len(res['energy_profile'])
    axes[0].plot(range(L), res['energy_profile'], 'o-', label=name, markersize=4)
    axes[1].plot(range(1, L), res['delta_h'], 'o-', label=name, markersize=4)

axes[0].set_ylabel('Mean H = T + V'); axes[0].set_xlabel('Layer')
axes[0].set_title('Energy Profile', fontweight='bold'); axes[0].legend(fontsize=8)
axes[1].set_ylabel('Mean |ΔH|'); axes[1].set_xlabel('Layer')
axes[1].set_title('Per-Layer Energy Change', fontweight='bold'); axes[1].legend(fontsize=8)

# Work decomposition bar chart
bar_width = 0.25
x_pos = np.arange(len(arm4_results[list(arm4_results.keys())[0]]['w_damp']))
for i, (name, res) in enumerate(arm4_results.items()):
    axes[2].bar(x_pos + i * bar_width, [abs(w) for w in res['w_exchange']],
                bar_width, label=f'{name} (exchange)', alpha=0.7)
axes[2].set_ylabel('|W_exchange|'); axes[2].set_xlabel('Layer')
axes[2].set_title('Exchange Work Budget', fontweight='bold')
axes[2].legend(fontsize=7)

for ax in axes: ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()
print('\nArm 4 complete. ✓')

In [ ]:
# ── Cell 8: Arm 5 — Conservativity Separator ──────────────────────
# Fit Δh = M·h per layer:
#   (a) unconstrained M         → R²_full
#   (b) symmetric M = (M+M^T)/2 → R²_sym
# Memory: light (CPU numpy only). One model at a time.

from sklearn.linear_model import LinearRegression

print('═' * 60)
print('ARM 5: Conservativity Separator')
print('═' * 60)

arm5_results = {}


def r2_full(X, Y):
    """R² for Y = M·X + b (unconstrained)."""
    return LinearRegression().fit(X, Y).score(X, Y)


def r2_sym(X, Y):
    """R² for Y ≈ M_sym·X where M_sym = (M+M^T)/2."""
    reg = LinearRegression(fit_intercept=False).fit(X, Y)
    M = reg.coef_
    M_sym = (M + M.T) / 2.0
    Y_pred = X @ M_sym.T
    ss_res = ((Y - Y_pred) ** 2).sum()
    ss_tot = ((Y - Y.mean(axis=0)) ** 2).sum()
    return float(1.0 - ss_res / max(ss_tot, 1e-10))


for name in MODEL_NAMES:
    print(f'\n── {name} ──')
    model, is_splm = load_model(name)
    n_eval = min(N_BATCHES, 3)
    r2f_accum, r2s_accum = None, None

    for bi in range(n_eval):
        x = get_batch_x(bi)
        traj, _ = extract_trajectory(model, x, is_splm)   # traj on CPU
        L = len(traj) - 1
        del x
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

        if r2f_accum is None:
            r2f_accum = [[] for _ in range(L)]
            r2s_accum = [[] for _ in range(L)]

        for ell in range(L):
            h_in  = traj[ell].float().numpy().reshape(-1, traj[ell].shape[-1])
            h_out = traj[ell+1].float().numpy().reshape(-1, traj[ell+1].shape[-1])
            dh = h_out - h_in

            n = min(len(h_in), 512)
            idx = np.random.choice(len(h_in), n, replace=False)
            X, Y = h_in[idx], dh[idx]

            r2f_accum[ell].append(r2_full(X, Y))
            r2s_accum[ell].append(r2_sym(X, Y))

        del traj

    mean_f = [np.mean(r) for r in r2f_accum]
    mean_s = [np.mean(r) for r in r2s_accum]

    arm5_results[name] = {
        'r2_full': mean_f, 'r2_sym': mean_s,
        'mean_r2_full': float(np.mean(mean_f)),
        'mean_r2_sym':  float(np.mean(mean_s)),
    }
    print(f'  R²_full per layer: {["{:.4f}".format(r) for r in mean_f]}')
    print(f'  R²_sym  per layer: {["{:.4f}".format(r) for r in mean_s]}')
    print(f'  Mean R²_full: {np.mean(mean_f):.4f}   Mean R²_sym: {np.mean(mean_s):.4f}')
    free_mem(model); del model

# ── Bar chart ──
fig, ax = plt.subplots(figsize=(8, 5))
names = list(arm5_results.keys())
r2f_vals = [arm5_results[n]['mean_r2_full'] for n in names]
r2s_vals = [arm5_results[n]['mean_r2_sym']  for n in names]
x_pos = np.arange(len(names))
w = 0.32
ax.bar(x_pos - w/2, r2f_vals, w, label='R² unconstrained', color='#3B82F6')
ax.bar(x_pos + w/2, r2s_vals, w, label='R² symmetric',     color='#22C55E')
ax.axhline(0.46, color='red', ls='--', alpha=0.5, label='GPT-2 ref (0.46)')
ax.set_xticks(x_pos); ax.set_xticklabels(names, fontsize=9)
ax.set_ylabel('R²'); ax.set_title('Conservativity Separator', fontweight='bold')
ax.legend(fontsize=8); ax.set_ylim(0, 1.05)
for i, (f, s) in enumerate(zip(r2f_vals, r2s_vals)):
    ax.text(i - w/2, f + 0.02, f'{f:.3f}', ha='center', fontsize=8)
    ax.text(i + w/2, s + 0.02, f'{s:.3f}', ha='center', fontsize=8)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()
print('\nArm 5 complete. ✓')

In [ ]:
# ── Cell 9: Summary Dashboard — 5-panel figure ────────────────────

print('═' * 60)
print('SUMMARY DASHBOARD')
print('═' * 60)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
C = {'Multi-Xi SPLM': '#3B82F6', 'Fock v2.1': '#22C55E', 'Fock Attention': '#F59E0B'}

# (a) Metric validity
ax = axes[0, 0]
for n, r in arm1_results.items():
    ax.plot(range(1, len(r['frac_positive'])+1), r['frac_positive'],
            'o-', color=C[n], label=n, markersize=4)
ax.set_title('(a) Metric Validity: Ω² > 0', fontweight='bold')
ax.set_xlabel('Layer'); ax.set_ylabel('Fraction Ω² > 0')
ax.set_ylim(0.9, 1.01); ax.legend(fontsize=7)

# (b) Geodesic compliance
ax = axes[0, 1]
for n, r in arm2_results.items():
    ax.plot(range(2, len(r['compliance'])+2), r['compliance'],
            'o-', color=C[n], label=n, markersize=4)
ax.set_title('(b) Geodesic Compliance (Jacobi R²)', fontweight='bold')
ax.set_xlabel('Layer'); ax.set_ylabel('Compliance')
ax.axhline(0, color='gray', ls='--', alpha=0.4); ax.legend(fontsize=7)

# (c) Curvature vs entropy
ax = axes[0, 2]
for n, r in arm3_results.items():
    ax.scatter(r['kmax'], r['entropy'], alpha=0.4, s=8, color=C[n],
               label=f"{n} (ρ={r['spearman_rho']:.2f})")
ax.set_title('(c) Curvature vs. Uncertainty', fontweight='bold')
ax.set_xlabel('K_max'); ax.set_ylabel('Logit entropy'); ax.legend(fontsize=7)

# (d) Energy profile
ax = axes[1, 0]
for n, r in arm4_results.items():
    ax.plot(range(len(r['energy_profile'])), r['energy_profile'],
            'o-', color=C[n], label=n, markersize=4)
ax.set_title('(d) Energy Profile H = T + V', fontweight='bold')
ax.set_xlabel('Layer'); ax.set_ylabel('Mean H'); ax.legend(fontsize=7)

# (e) Exchange work budget
ax = axes[1, 1]
for n, r in arm4_results.items():
    ax.plot(range(1, len(r['w_exchange'])+1),
            [abs(w) for w in r['w_exchange']],
            'o-', color=C[n], label=n, markersize=4)
ax.set_title('(e) Exchange Work |W_ex|', fontweight='bold')
ax.set_xlabel('Layer'); ax.set_ylabel('|W_exchange|'); ax.legend(fontsize=7)

# (f) Conservativity separator
ax = axes[1, 2]
names_list = list(arm5_results.keys())
r2s = [arm5_results[n]['mean_r2_sym'] for n in names_list]
bars = ax.bar(range(len(names_list)), r2s,
              color=[C[n] for n in names_list])
ax.axhline(0.46, color='red', ls='--', alpha=0.5, label='GPT-2 (0.46)')
ax.set_title('(f) Conservativity Separator (R² sym)', fontweight='bold')
ax.set_xticks(range(len(names_list))); ax.set_xticklabels(names_list, fontsize=8)
ax.set_ylabel('R²'); ax.set_ylim(0, 1.05); ax.legend(fontsize=7)
for i, v in enumerate(r2s):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')

for row in axes:
    for a in row:
        a.spines['top'].set_visible(False)
        a.spines['right'].set_visible(False)

plt.suptitle('Riemannian Geometry Diagnostic Battery — SPLM Model Family',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
DASH_PATH = 'riemannian_diagnostic_dashboard.png'
fig.savefig(DASH_PATH, dpi=200, bbox_inches='tight', facecolor='white')
plt.show()

# ── Summary table ──
print('\n┌────────────────────┬──────────┬──────────┬──────────┐')
print('│ Metric             │ SPLM     │ Fock v2.1│ Fock Attn│')
print('├────────────────────┼──────────┼──────────┼──────────┤')
s1, f1, a1 = [arm1_results[n] for n in ['Multi-Xi SPLM','Fock v2.1','Fock Attention']]
print(f'│ Ω²>0 fraction (min)│ {min(s1["frac_positive"]):8.4f} │ {min(f1["frac_positive"]):8.4f} │ {min(a1["frac_positive"]):8.4f} │')
s2, f2, a2 = [arm2_results[n] for n in ['Multi-Xi SPLM','Fock v2.1','Fock Attention']]
print(f'│ Geodesic R² (μ)    │ {s2["mean_compliance"]:8.4f} │ {f2["mean_compliance"]:8.4f} │ {a2["mean_compliance"]:8.4f} │')
s3, f3, a3 = [arm3_results[n] for n in ['Multi-Xi SPLM','Fock v2.1','Fock Attention']]
print(f'│ ρ(K_max, entropy)  │ {s3["spearman_rho"]:8.4f} │ {f3["spearman_rho"]:8.4f} │ {a3["spearman_rho"]:8.4f} │')
s4, f4, a4 = [arm4_results[n] for n in ['Multi-Xi SPLM','Fock v2.1','Fock Attention']]
print(f'│ Energy drift (%)   │ {s4["relative_drift"]*100:7.2f}% │ {f4["relative_drift"]*100:7.2f}% │ {a4["relative_drift"]*100:7.2f}% │')
s5, f5, a5 = [arm5_results[n] for n in ['Multi-Xi SPLM','Fock v2.1','Fock Attention']]
print(f'│ R²_sym (mean)      │ {s5["mean_r2_sym"]:8.4f} │ {f5["mean_r2_sym"]:8.4f} │ {a5["mean_r2_sym"]:8.4f} │')
print('└────────────────────┴──────────┴──────────┴──────────┘')

In [ ]:
# ── Cell 10: Save results to GDrive ───────────────────────────────
import json, shutil

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/semsimula_riemannian_diagnostic'
else:
    SAVE_DIR = 'results/riemannian_diagnostic'

os.makedirs(SAVE_DIR, exist_ok=True)

report = {
    'experiment': 'riemannian_geometry_diagnostic_battery',
    'models': list(MODELS.keys()),
    'config': {
        'n_batches': N_BATCHES, 'batch_size': BATCH_SIZE,
        'block_size': BLOCK_SIZE, 'n_power_iter': N_POWER_ITER,
        'n_sample_positions': N_SAMPLE_POS,
        'n_sample_sequences': N_SAMPLE_SEQ, 'device': DEVICE,
    },
    'arm1_metric_validity': arm1_results,
    'arm2_geodesic_compliance': arm2_results,
    'arm3_curvature_proxy': arm3_results,
    'arm4_energy_conservation': arm4_results,
    'arm5_conservativity_separator': arm5_results,
}

rpt_path = os.path.join(SAVE_DIR, 'riemannian_diagnostic_report.json')
with open(rpt_path, 'w') as f:
    json.dump(report, f, indent=2, default=str)
print(f'Report: {rpt_path}')

if os.path.exists(DASH_PATH):
    dst = os.path.join(SAVE_DIR, 'riemannian_diagnostic_dashboard.png')
    shutil.copy2(DASH_PATH, dst)
    print(f'Dashboard: {dst}')

print(f'\nAll results persisted to: {SAVE_DIR}')
print('\n' + '═' * 60)
print('RIEMANNIAN GEOMETRY DIAGNOSTIC BATTERY — COMPLETE')
print('═' * 60)